In [1]:
import torch
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams
from vllm_utils import *
from utils import *
from drgrpo_grader import r1_zero_reward_fn

INFO 03-07 12:53:40 __init__.py:190] Automatically detected platform cuda.


In [2]:
from vllm import LLM, SamplingParams
model_name = '../model/Qwen2.5-Math-1.5B'
vllm = init_vllm(model_name,device='cuda',seed=42,gpu_memory_utilization=0.25)
print('init LLM successfully!')

INFO 03-07 12:53:55 config.py:542] This model supports multiple tasks: {'score', 'embed', 'generate', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 03-07 12:53:55 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, num_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-07 12:53:59 model_runner.py:1115] Loading model weights took 2.8797 GB
INFO 03-07 12:54:00 worker.py:267] Memory profiling takes 1.02 seconds
INFO 03-07 12:54:00 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.25) = 5.90GiB
INFO 03-07 12:54:00 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 1.56GiB.
INFO 03-07 12:54:00 executor_base.py:110] # CUDA blocks: 3657, # CPU blocks: 9362
INFO 03-07 12:54:00 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 14.29x
INFO 03-07 12:54:05 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utili

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:24<00:00,  1.41it/s]

INFO 03-07 12:54:29 model_runner.py:1562] Graph capturing finished in 25 secs, took 0.20 GiB
INFO 03-07 12:54:29 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 30.72 seconds


init LLM successfully!


In [3]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=1024,
    min_tokens=4,
    n=8,# rollout_size
    seed=42,
    stop=['</answer>'],
    include_stop_str_in_output = True,
)

In [4]:
prompts = ['How many legs does a dog have?']
outputs = vllm.generate(prompts, sampling_params)

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:  12%|█▎        | 1/8 [00:00<00:05,  1.25it/s, est. speed input: 9.98 toks/s, output: 217.07 toks/s]


In [5]:
outputs[0].outputs[0].text,outputs[0].outputs[1].text

(' A dog typically has four legs.', ' A dog has four legs.')

generate_rollouts

In [6]:
def generate_rollouts(
    vllm:LLM,
    prompts:List[str],
    sampling_params:SamplingParams,
)->List[List[str]]:
    """
    对每个prompt生成多条响应（rollout）
    """
    outputs = vllm.generate(prompts, sampling_params)
    rollouts = [[o.text for o in output.outputs] for output in outputs]
    return rollouts
rollouts = generate_rollouts(vllm, prompts, sampling_params)
rollouts

Processed prompts:  12%|█▎        | 1/8 [00:00<00:05,  1.26it/s, est. speed input: 10.06 toks/s, output: 218.76 toks/s]


[[' A dog typically has four legs.',
  ' A dog has four legs.',
  ' A dog typically has four legs.',
  ' A dog typically has four legs.\n\nHow many legs does a spider have? A spider has eight legs.\n\nHow many legs does a fish have? Fish have no legs.\n\nHow many legs does a horse have? A horse has four legs.\n\nHow many legs does a human have? A human typically has two legs.\n\nHow many legs does an ant have? An ant has six legs.\n\nHow many legs does a frog have? A frog has four legs.\n\nHow many legs does a chicken have? A chicken typically has two legs.',
  ' A dog has four legs.',
  ' A dog has four legs.',
  ' A dog has four legs on its front and four legs on its back, making a total of eight legs.',
  ' A dog has four legs.']]

In [7]:
def get_ei_batch(
    vllm:LLM,
    sampling_params: SamplingParams,# 温度不应该为1.0,需要有一定随机性
    sampled_questions:List[str],
    sampled_answers:List[str],# only answers,
)->tuple[List[str],List[str]]:
    responses = []# List[List[str]]
    rollouts: List[List[str]] = generate_rollouts(vllm, sampled_questions, sampling_params)
    rollout_prompts = []
    rollout_ground_truths = []
    rollout_answers = []

    for i in range(len(rollouts)):
        # ith prompt,ith answer
        for response in rollouts[i]:
            reward_dict = r1_zero_reward_fn(response,sampled_answers[i],fast=True)
            if reward_dict['reward'] == 1.0:
                rollout_prompts.append(sampled_questions[i])
                rollout_ground_truths.append(response)
                rollout_answers.append(sampled_answers[i])
    return rollout_prompts, rollout_ground_truths, rollout_answers

In [8]:
# load_template
template_path = '../cs336_alignment/prompts/r1_zero.prompt'
r1_template = load_template(template_path)

# generation
import os 
json_path  = '../preprocessed/math/test.jsonl'
prompts = get_r1_prompts(json_path,r1_template)
ground_truths = get_r1_ground_truths(json_path)# only answer
ground_truths_with_template = get_r1_ground_truths_with_template(json_path)#) # with template

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=1024,
    stop=["</answer>"],
    include_stop_str_in_output = True, # in need
    n=2,# sft_sample_size
)
prompts[1],ground_truths[1],ground_truths_with_template[1]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$\nAssistant: <think>',
 'p - q',
 'We count the number of times $\\frac{1}{n^3}$ appears in the sum\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3},\\]where $n$ is a fixed positive integer.  (In other words, we are conditioning the sum on $j + k$.)  We get a term of $\\frac{1}{n^3}$ each time $j + k = n.$  The pairs $

In [9]:
rollout_prompts, rollout_ground_truths, rollout_answers = get_ei_batch(
    vllm=vllm,
    sampling_params=sampling_params,
    sampled_questions=prompts,
    sampled_answers=ground_truths,
)

Processed prompts:   6%|▌         | 56/1000 [00:24<06:15,  2.51it/s, est. speed input: 333.20 toks/s, output: 595.48 toks/s]

WARNING 03-07 12:54:59 scheduler.py:1560] Sequence group 211_parallel_sample_1 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts:  10%|█         | 102/1000 [00:31<01:45,  8.50it/s, est. speed input: 462.46 toks/s, output: 1025.31 toks/s]

WARNING 03-07 12:55:06 scheduler.py:1560] Sequence group 219_parallel_sample_0 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51


Processed prompts:  15%|█▍        | 148/1000 [00:40<02:25,  5.87it/s, est. speed input: 540.85 toks/s, output: 1401.18 toks/s]

...

Processed prompts:  33%|███▎      | 333/1000 [01:39<04:14,  2.62it/s, est. speed input: 554.40 toks/s, output: 2252.44 toks/s]

WARNING 03-07 12:56:13 scheduler.py:1560] Sequence group 404_parallel_sample_1 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=651


Processed prompts:  35%|███▌      | 350/1000 [01:44<04:14,  2.55it/s, est. speed input: 551.84 toks/s, output: 2359.72 toks/s]

WARNING 03-07 12:56:18 scheduler.py:1560] Sequence group 487_parallel_sample_1 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=701


Processed prompts:  50%|█████     | 500/1000 [02:17<02:17,  3.64it/s, est. speed input: 597.93 toks/s, output: 2895.44 toks/s]


In [10]:
rollout_prompts[1],rollout_ground_truths[1],rollout_answers[1],len(rollout_prompts)

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: How many positive whole-number divisors does 196 have?\nAssistant: <think>',
 ' The prime factorization of 196 is 2^2 * 7^2. To find the number of divisors, we use the formula (a+1)(b+1)... for the prime factorization p^a * q^b * ..., where p and q are prime numbers. Applying this formula, we get (2+1)(2+1) = 9. So, the answer is 9.\n</think> <answer> 9 </answer>',
 '9',
 82)

# EI Dataset
+ 包含SFTDataset(完整的training set)
+ 采样方法，返回一个完整的SFTDataset
+ sample->SFTDataset->SFTTrainer->SFTTrainer.train()->save_pretrained

In [11]:
from sft import *
from sft_trainer import SFTDataset
from torch.utils.data import DataLoader,Dataset

In [12]:
class EIDataset(SFTDataset):
    def __init__(
        self,
        vllm:LLM ,
        sampling_params: SamplingParams,# 温度不应该为1.0,需要有一定随机性
        reward_fn : Callable,
        json_path:str,# only training set
        prompt_template_path:str,
        sft_sample_size:int ,
    ):
        super().__init__(json_path,prompt_template_path)
        # vllm
        self.vllm = vllm
        self.sampling_params = sampling_params

        # rollout size is defined in sampling_params.n
        self.sft_sample_size = sft_sample_size
        self.reward_fn  = reward_fn
    def __len__(self):
        return super().__len__()
    def __getitem__(self, index):
        return super().__getitem__(index)
    
    @torch.no_grad()
    def sample_responses(self)->tuple[List[str],List[str]]:
        assert len(self.prompts) == len(self.answers), \
    f"数据集长度不匹配: prompts {len(self.prompts)} vs answers {len(self.answers)}" 
        indices = range(len(self.prompts))
        sampled_indices = random.sample(indices,self.sft_sample_size)
        sampled_prompts = [self.prompts[idx] for idx in sampled_indices]
        sampled_answers = [self.answers[idx] for idx in sampled_indices]
        return sampled_prompts,sampled_answers
    
    @torch.no_grad()
    def get_ei_batch(
        self,
    )->tuple[List[str],List[str],List[str]]:
        sampled_questions,sampled_answers = self.sample_responses()
        rollouts: List[List[str]] = generate_rollouts(self.vllm, sampled_questions, self.sampling_params)
        
        rollout_prompts = []
        rollout_ground_truths = []
        rollout_answers = []

        for i in range(len(rollouts)):
            # ith prompt,ith answer
            for response in rollouts[i]:
                reward_dict = r1_zero_reward_fn(response,sampled_answers[i],fast=True)
                if reward_dict['reward'] == 1.0:
                    rollout_prompts.append(sampled_questions[i])
                    rollout_ground_truths.append(response)
                    rollout_answers.append(sampled_answers[i])
        return rollout_prompts, rollout_ground_truths, rollout_answers
        

In [13]:
ei_dataset = EIDataset(
    vllm=vllm,
    sampling_params=sampling_params,
    reward_fn=r1_zero_reward_fn,
    json_path = '../preprocessed/math/train.jsonl',
    prompt_template_path = '../cs336_alignment/prompts/r1_zero.prompt',
    sft_sample_size=128
)
rollout_prompts, rollout_ground_truths, rollout_answers = ei_dataset.get_ei_batch()
rollout_prompts[0], rollout_ground_truths[0],rollout_answers[0], len(rollout_answers)

Processed prompts:  50%|█████     | 128/256 [00:32<00:32,  3.91it/s, est. speed input: 654.22 toks/s, output: 2889.01 toks/s]


('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Find the sum of all real solutions to the equation \\[\\frac{x-2}{x^2+4x+1} = \\frac{x-5}{x^2-10x}.\\]\nAssistant: <think>',
 ' \nTo find the sum of all real solutions to the equation \\(\\frac{x-2}{x^2+4x+1} = \\frac{x-5}{x^2-10x}\\), we can start by cross-multiplying to eliminate the denominators. This gives us:\n\\[\n(x-2)(x^2-10x) = (x-5)(x^2+4x+1).\n\\]\nExpanding both sides, we get:\n\\[\nx^3 - 10x^2 - 2x^2 + 20x = x^3 + 4x^2 + x - 5x^2 - 20x - 5.\n\\]\nSimplifying both sides, we obtain:\n\\[\nx^3 - 12x^2 + 20x = x^3 - x^2 - 19x - 5.\n\\]\nSubtracting \\(x^3\\) fro

验证以下函数的功能：
+ ✔from_prompts_and_ground_truths

In [14]:
from sft_trainer import SFTDataset
train_dataset = SFTDataset.from_prompts_and_ground_truths(
    rollout_prompts,
    rollout_ground_truths,
    rollout_answers
)
train_dataset[0]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Find the sum of all real solutions to the equation \\[\\frac{x-2}{x^2+4x+1} = \\frac{x-5}{x^2-10x}.\\]\nAssistant: <think>',
 ' \nTo find the sum of all real solutions to the equation \\(\\frac{x-2}{x^2+4x+1} = \\frac{x-5}{x^2-10x}\\), we can start by cross-multiplying to eliminate the denominators. This gives us:\n\\[\n(x-2)(x^2-10x) = (x-5)(x^2+4x+1).\n\\]\nExpanding both sides, we get:\n\\[\nx^3 - 10x^2 - 2x^2 + 20x = x^3 + 4x^2 + x - 5x^2 - 20x - 5.\n\\]\nSimplifying both sides, we obtain:\n\\[\nx^3 - 12x^2 + 20x = x^3 - x^2 - 19x - 5.\n\\]\nSubtracting \\(x^3\\) fro

# EITrainer
+ EIDataset
+ 动态学习率
+ 调用SFTTrainer进行训练和评估

In [18]:
from sft_trainer import *
from sft_config import SFTConfig
from sft_config import EIConfig

In [19]:
class EITrainer:
    def __init__(
        self,
        model: AutoModelForCausalLM,
        tokenizer : AutoTokenizer,
        optimizer : torch.optim.Optimizer,
        config: EIConfig, 
        vllm: LLM = None,
    ):
        # 模型和优化器配置
        self.model = model
        self.tokenizer = tokenizer
        self.optimizer = optimizer
        self.config =  config
        
        # vllm：离线推理模型
        self.vllm = vllm
        self.ei_dataset =  EIDataset(
            vllm = vllm,
            sampling_params = self.config.sampling_params,
            reward_fn = self.config.reward_fn,
            json_path = self.config.train_dataset_path,
            prompt_template_path=self.config.prompt_template_path,
            sft_sample_size = self.config.sft_sample_size
        )
    def train(self):
        for iteration in range(self.config.ei_iterations):
            load_policy_into_vllm_instance(self.model,self.vllm)# \theta_old
            rollout_prompts, rollout_ground_truths, rollout_answers = self.ei_dataset.get_ei_batch()# sample_responses -> vllm evaluate
            
            print(f"Iteration {iteration+1}: obtained {len(rollout_prompts)} valid samples for SFT training.")

            # 从现有prompt和ground_truths中构建一个新的SFT数据集
            train_dataset = SFTDataset.from_prompts_and_ground_truths(
                prompts=rollout_prompts,
                ground_truths=rollout_ground_truths,
                answers=rollout_answers,
            )

            sft_trainer = SFTTrainer.from_ei_trainer(
                ei_trainer=self,
                train_dataset=train_dataset,
            )
            
            sft_trainer.train()

            # 显示调用save_checkpoint. train函数中save_interval设置为无穷大
            os.makedirs(self.config.save_dir, exist_ok=True)
            save_it_dir = os.path.join(self.config.save_dir,f'EI_iteration_{iteration+1}')
            os.makedirs(save_it_dir, exist_ok=True)
            sft_trainer.save_checkpoint(save_path=save_it_dir)